In [ ]:
import serial
import csv
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- CELL 1: Automated Sweep and Data Collection ---
def run_automated_sweep(port, baudrate, filename, power_levels, duration_per_level):
    """
    Connects to the ESP32, runs through a sequence of PWM power levels,
    records the resulting telemetry, and ensures a safe spin-down.
    """
    print(f"Connecting to {port} at {baudrate} baud...")
    try:
        ser = serial.Serial(port, baudrate, timeout=0.1)
        time.sleep(2) # Give ESP32 time to reset upon connection
    except Exception as e:
        print(f"Failed to connect: {e}")
        return

    print("Starting automated test sweep...")
    
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Timestamp_us", "Command_PWM", "Accel_X", "Accel_Y", "Accel_Z", "RPM"])
        
        try:
            for power in power_levels:
                print(f"Setting power level: {power}us for {duration_per_level} seconds.")
                start_time = time.time()
                last_ping_time = 0
                
                # Run this power level for the specified duration
                while (time.time() - start_time) < duration_per_level:
                    
                    # Ping the ESP32 every 0.25 seconds to keep the failsafe from triggering
                    if (time.time() - last_ping_time) > 0.25:
                        ser.write(f"{power}\n".encode('utf-8'))
                        last_ping_time = time.time()

                    # Check for and record incoming data
                    if ser.in_waiting > 0:
                        try:
                            line = ser.readline().decode('utf-8').strip()
                            data = line.split(',')
                            if len(data) == 6: # Ensure we have all 6 columns
                                writer.writerow(data)
                        except Exception:
                            pass # Ignore malformed packets 
                            
        except KeyboardInterrupt:
            print("\nSweep interrupted by user!")
            
        finally:
            print("Test complete. Spinning down...")
            # Send stop command several times to guarantee delivery
            for _ in range(5):
                ser.write(b"1000\n")
                time.time()
            ser.close()
            print(f"Data successfully saved to {filename}")




# --- CELL 2: Data Processing and Polynomial Modeling ---
def generate_rpm_model(csv_filename, degree=2):
    df = pd.read_csv(csv_filename)
    
    # Isolate strictly the periods where the robot was spinning 
    # (Drop data where PWM command was 1000/idle)
    df = df[df['Command_PWM'] > 1000]
    
    accel = df['Accel_Y'].abs().values 
    rpm = df['RPM'].values
    
    df['RPM_Smooth'] = df['RPM'].rolling(window=10).mean()
    df['Accel_Y_Smooth'] = df['Accel_Y'].abs().rolling(window=10).mean()
    df = df.dropna()
    
    accel_clean = df['Accel_Y_Smooth'].values
    rpm_clean = df['RPM_Smooth'].values
    
    coefficients = np.polyfit(accel_clean, rpm_clean, degree)
    poly_func = np.poly1d(coefficients)
    
    x_fit = np.linspace(min(accel_clean), max(accel_clean), 100)
    y_fit = poly_func(x_fit)
    
    plt.figure(figsize=(10, 6))
    
    # Color-code the scatter plot based on the commanded PWM
    scatter = plt.scatter(accel_clean, rpm_clean, c=df['Command_PWM'], cmap='viridis', alpha=0.5, label='Raw Data (Smoothed)')
    plt.colorbar(scatter, label='Commanded PWM (µs)')
    
    plt.plot(x_fit, y_fit, color='red', linewidth=3, label=f'Poly Fit (Degree {degree})')
    
    plt.title('Meltybrain Radial Acceleration (Y-Axis) vs. Ground Truth RPM')
    plt.xlabel('Radial Acceleration (G)')
    plt.ylabel('Rotational Velocity (RPM)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print("Model Coefficients (Highest to lowest degree):")
    print(coefficients)
    print("\nC++ Implementation formula:")
    
    cpp_formula = "float predicted_rpm = "
    for i, coef in enumerate(coefficients):
        power = degree - i
        if power == 0:
            cpp_formula += f"({coef});"
        else:
            cpp_formula += f"({coef} * pow(abs(yAccel), {power})) + "
            
    print(cpp_formula)
    
# Example usage:
# generate_rpm_model('melty_sweep_1.csv', degree=2)

In [ ]:
sweep_profile = [1050, 1100, 1150, 1200, 1250, 1200, 1100, 1000]
run_automated_sweep('COM3', 115200, 'melty_sweep_1.csv', sweep_profile, duration_per_level=5.0)